# Lab Implementation of simple RNN and LSTM 

This notebook has been prepared by Hsiu-Wen Chang from MINES ParisTech
Shall you have any problem, send me [email](hsiu-wen.chang_joly@mines-paristech.fr)

## Learning objectives

1. many-to-one by RNN: given several words, predict the next word
2. many-to-many(sequence to sequence) by moduled PyTorch funtions: given a English sentence, translate to French sentence
3. Tokenize sentence
4. Numerize tokens

## 1. Many-to-one by RNN (word level): Predict what is the next word

Our task today is to predict the next word by given several words before. For example, we expect to have answer to be 'cat' when user key in 'I like'.

In [ ]:
# Configuration
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

dtype = torch.float
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


#### 1.1 Data preparation

Here are three sentences and each of them has three words. We are going to use it as training sample. The design is to feed first two words and let the machine find the final word. However, the computer can't do mathematic operations on characters. Therefore, the first step is to encode the input to digital numbers. 

In [ ]:
# Create the input data, you are welcome to add the words you like
sentences = [ "i like cat", "i love coffee", "i hate milk"]

# Define all the possible words
word_list = " ".join(sentences).split()

word_list = list(set(word_list))

# dictionary that chanage the given word to number. {love: 0, hate:1,...}
word_dict = {w: i for i, w in enumerate(word_list)}

# dictionary that chanage the number to word. {0: love, 1: hate,...}
number_dict = {i: w for i, w in enumerate(word_list)}

# number of class(=number of vocab)
n_class = len(word_dict)

print(word_dict)

#### 1.2 Data preprocessing

Define batch function to let machine know how he should use it during training.
Here we give all the data we have for simplication. But in real case, you should not do it.


In [ ]:
# Function to encode the sentence into a vector 
def make_batch(sentences):
    input_batch = []
    target_batch = []

    for sen in sentences:
        word = sen.split()
        input = [word_dict[n] for n in word[:-1]]
        target = word_dict[word[-1]]

        input_batch.append(np.eye(n_class)[input])
        target_batch.append(np.eye(n_class)[target])

    return input_batch, target_batch

In [ ]:
input_batch, target_batch = make_batch(sentences)
# move input and output to the device
input_batch = torch.tensor(input_batch, device=device, dtype=dtype)
target_batch = torch.tensor(target_batch, device=device, dtype=dtype)

print('Dimension of input_patch:', input_batch.shape) #[batch, N_step, n_class]
print(input_batch)
print('Dimension of target_batch:', target_batch.shape) #[batch, N_step, n_class]
print(target_batch)

**Question**: How do you make a batch with different lenghth of sentences? 

#### 1.3 Define your first RNN Network

As represented in lecture, the equations of vanilla RNN are 
$$
h_t=tanh(W_{ih}x_t+b_{ih}+w_{hh}h_{t-1}+b_{hh}) \tag{1}
$$
$$
y_t = W_{hy}h_t+b_{y} \tag{2}
$$

In [ ]:
class TextRNN(nn.Module):
    def __init__(self,n_class=7, n_hidden=5):
        super(TextRNN, self).__init__()
        
        # Parameters of the RNN
        self.W_xh = nn.Parameter(torch.randn(n_class, n_hidden) * 0.1)
        self.W_hh = nn.Parameter(torch.randn(n_hidden, n_hidden) * 0.1)
        self.b_h = nn.Parameter(torch.zeros(n_hidden))
        #self.rnn = nn.RNN(input_size=n_class, hidden_size=n_hidden)
        self.W_hy = nn.Parameter(torch.randn([n_hidden, n_class]) * 0.1)
        self.b_y = nn.Parameter(torch.randn([n_class]))

    def forward(self, X):
        batch_size, n_step, n_class = X.size()
        # initialize the hidden state
        h=torch.zeros(batch_size, self.W_hh.size(0), device=X.device)
        # outputs(batch, n_step, n_class)
        outputs = []
        for t in range(n_step):
            x_t = X[:, t, :]
            h = torch.tanh(x_t @ self.W_xh + h @ self.W_hh + self.b_h)
            y_t = h @ self.W_hy + self.b_y
            outputs.append(y_t.unsqueeze(1))
     
        return torch.cat(outputs, dim=1), h

In [ ]:
# Intialize the model and move it to the device
model = TextRNN(n_class=n_class, n_hidden=5).to(device)
# show the training parameters of this model
print(list(model.parameters()))

**Question** What are the meaning of these parameters? Can you identify which one eqaul to the provided training paramters in the equations (1) and (2)?

#### 1.4 Training

In [ ]:
# Hyperparameters
learning_rate = 0.001
batch_size = len(sentences)
n_step = 2
n_hidden = 5
training_epochs = 5000

Define loss function and algorithm of optimizer, here we try famous Adam algorithm

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(training_epochs):
    # Reset the gradient buffer 
    optimizer.zero_grad()
    
    # Forward pass
    outputs, hidden = model(input_batch)
    
    # Compute the loss by comaring the last output of the RNN with the target batch
    loss = criterion(outputs[:, -1].view(-1, n_class), target_batch.view(-1, n_class))
    
    # Backward pass and optimization
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 1000 == 0:
        print(f'Epoch [{epoch+1}/{training_epochs}], Loss: {loss.item():.4f}')

#### 1.5 Test the model

In [ ]:
raw_output, _ = model(input_batch)
print('Raw output of this model (batch,n_step, n_class):\n', raw_output)
# We use the last n_step output of the RNN to predict the target word
predict = raw_output[:, -1].max(1, keepdim=True)[1]
print([sen.split()[:2] for sen in sentences], '->', [number_dict[n.item()] for n in predict.squeeze()])#

**Question**: So far, we have seen how to **generate** a word during the final step. What are the words predicted prior to this final step? Do they convey meaning? When we translate French sentences to English sentences. What are the expected modification for the following functions? 

- num_dir:
- make_batch:

Waht are the issue of number of output tensors?


## 2. Training a bigger language dataset with vary lengths

As the previous task demonstrates, it is impossible to achieve high precision if the number of English words differs from the number of French words. In this case, we must **summarize** the input and use additional tokens to delimit the beginning and the end. This is the objective of this section. 

### 2.1 Data preparation and preprocessing

To increase the difficuty, we use the English–French corpus from the [D2L machine-translation lesson](https://d2l.ai/chapter_recurrent-modern/machine-translation-and-dataset.html). It contains tab-delimited English/French sentence pairs from the [Tatoeba Project](https://www.manythings.org/anki/). Each line in the dataset is a tab-delimited pair consisting of an English text sequence (the source) and the translated French text sequence (the target). To keep the practical short, we train on a filtered sample of 800 pairs.

Function `preprocess` will replace non-breaking space with space, convert uppercase letters to lowercase one, and insert space between words and punctuation marks.

In [ ]:
import random
import re
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
DATA_URL = 'http://d2l-data.s3-accelerate.amazonaws.com/fra-eng.zip'
data_dir = Path('data')
archive_path = data_dir / 'fra-eng.zip'
text_path = data_dir / 'fra-eng' / 'fra.txt'
data_dir.mkdir(exist_ok=True)

if not text_path.exists():
    if not archive_path.exists():
        print('Downloading English–French corpus...')
        urllib.request.urlretrieve(DATA_URL, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(data_dir)

def preprocess(text):
    text = text.replace('\u202f', ' ').replace('\xa0', ' ').lower()
    return re.sub(r'([,.!?])', r' \1 ', text)

def load_pairs(path, max_pairs=800, max_tokens=8):
    result = []
    for line in path.read_text(encoding='utf-8').splitlines():
        fields = line.split('\t')
        if len(fields) < 2:
            continue
        english, french = (preprocess(fields[0]), preprocess(fields[1]))
        if 0 < len(english.split()) <= max_tokens and 0 < len(french.split()) <= max_tokens:
            result.append((english, french))
        if len(result) == max_pairs:
            break
    return result

pairs = load_pairs(text_path)
random.shuffle(pairs)
split = int(0.85 * len(pairs))
train_pairs, test_pairs = pairs[:split], pairs[split:]
print(f'{len(train_pairs)} training pairs, {len(test_pairs)} test pairs')
train_pairs[:4]

### 2.2. Tokenisation and vocabularies

In machine tranlation, target and source sentences may have different lengths. For computational efficiency, a minibatch of these sequences is processed at one time by truncation (longer than num_steps) and padding (shorter than num_steps). We also need a token to indicate the end of the sentence and a token to indicate the start of the sentence. Since the vocalubary size is significatly large, words that appear less than twice will be treated as the same unknown token. Totally, we introduce four new tokens: 
 - padding `<pad>`
 - the beginning of a sentence `<sos>`
 - the end of a sentence`<eos>`
 - unknown `<unk>`.

 `tokenize` split a sentence to a set of words

 `build_vocab` create a vocabulary to include all the possible token and its index

 `numericalize` covert tokens to interger. 

In [ ]:
PAD, SOS, EOS, UNK = '<pad>', '<sos>', '<eos>', '<unk>'

def tokenize(text):
    return text.split()

def build_vocab(sentences):
    counts = Counter(token for sentence in sentences for token in tokenize(sentence))
    tokens = [PAD, SOS, EOS, UNK] + sorted(counts)
    stoi = {token: i for i, token in enumerate(tokens)}
    return stoi, tokens

src_stoi, src_tokens = build_vocab([en for en, _ in train_pairs])
tgt_stoi, tgt_tokens = build_vocab([fr for _, fr in train_pairs])
print(f'English vocabulary: {len(src_tokens)} tokens')
print(f'French vocabulary:  {len(tgt_tokens)} tokens')
print(src_stoi)
print(tgt_stoi)

In [ ]:
def numericalize(text, stoi, add_sos=False, add_eos=True):
    ids = [stoi.get(token, stoi[UNK]) for token in tokenize(text)]
    if add_sos:
        ids = [stoi[SOS]] + ids
    if add_eos:
        ids = ids + [stoi[EOS]]
    return torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(1)  # (length, batch=1)

example_en, example_fr = train_pairs[0]
print(example_en, '→', numericalize(example_en, src_stoi))
print(example_fr, '→', numericalize(example_fr, tgt_stoi, add_sos=True))

### 2.3 The encoder–decoder RNN

The **encoder** reads all English tokens and returns a final hidden state—a learned summary of the input. The **decoder** starts from that state and emits French tokens one at a time. We use a GRU, a gated RNN that helps preserve information over longer sequences.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.GRU(embedding_dim, hidden_dim)

    def forward(self, source):
        embedded = self.embedding(source)
        _, hidden = self.rnn(embedded)
        return hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.GRU(embedding_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token, hidden):
        embedded = self.embedding(token)
        output, hidden = self.rnn(embedded, hidden)
        return self.output(output.squeeze(0)), hidden

EMBEDDING_DIM, HIDDEN_DIM = 32, 64
encoder = Encoder(len(src_tokens), EMBEDDING_DIM, HIDDEN_DIM).to(device)
decoder = Decoder(len(tgt_tokens), EMBEDDING_DIM, HIDDEN_DIM).to(device)
print(f'Trainable parameters: {sum(p.numel() for p in encoder.parameters()) + sum(p.numel() for p in decoder.parameters()):,}')

### 2.4 Training with teacher forcing

During training, the decoder receives the *correct* previous French word as its next input. This is called teacher forcing. At inference time it has to use its own previous prediction instead.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)

def train_step(english, french):
    encoder.train(); decoder.train()
    source = numericalize(english, src_stoi)
    target = numericalize(french, tgt_stoi, add_sos=True)
    optimizer.zero_grad()
    hidden = encoder(source)
    loss = 0
    decoder_input = target[0:1]  # <sos>
    for step in range(1, len(target)):
        logits, hidden = decoder(decoder_input, hidden)
        loss = loss + criterion(logits, target[step])
        decoder_input = target[step:step + 1]  # teacher forcing
    loss.backward()
    torch.nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(decoder.parameters()), 1.0)
    optimizer.step()
    return loss.item() / (len(target) - 1)

losses = []
for epoch in range(1, 16):
    random.shuffle(train_pairs)
    epoch_loss = sum(train_step(en, fr) for en, fr in train_pairs) / len(train_pairs)
    losses.append(epoch_loss)
    if epoch % 3 == 0:
        print(f'Epoch {epoch:3d} | mean token loss = {epoch_loss:.4f}')

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Mean cross-entropy loss')
plt.title('Training curve')
plt.grid(alpha=0.3)
plt.show()

### 2.4 Mini batch

The aboved training cell did not use a batch of sentences to train this model. Read the build_arrays function and use it to answer question:

In [ ]:
# Build a mini batch of train pairs
def build_arrays(selected_pairs, src_vocab, tgt_vocab, num_steps=8):
    # Input: 
    #       a list of pairs of sentences
    #       the source and target vocabularies
    #       the number of steps (max length) for the source
    #       target sequences.
    # Output: 
    #       the source sequences
    #       the target sequences (excluding the last token)
    #       the valid lengths of the source sequences
    #       the target sequences (excluding the first token).
    def build_array(sentences, vocab, max_len, add_sos=False, add_eos=True):
        # Input: 
        #   a list of sentences, 
        #   a vocabulary
        #   the maximum length
        #   flags to add start-of-sequence and end-of-sequence tokens.
        # Output: 
        #   a tensor of shape (batch_size, max_len) containing the token IDs of the sentences
        #   a tensor of shape (batch_size,) containing the valid lengths of the sequences
        sequences = []

        for sentence in sentences:
            tokens = list(sentence)

            if add_sos:
                tokens.insert(0, SOS)
            if add_eos:
                tokens.append(EOS)

            # Trim sequence if it exceeds max_len
            tokens = tokens[:max_len]
            # Pad sequence if it is shorter than max_len
            tokens += [PAD] * (max_len - len(tokens))

            sequences.append([
                vocab.get(token, vocab[UNK])
                for token in tokens
            ])

        result = torch.tensor(sequences, dtype=torch.long, device=device)
        valid_len = (result != vocab[PAD]).sum(dim=1)

        return result, valid_len

    src_array, src_valid_len = build_array(
        [tokenize(en) for en, _ in selected_pairs],
        src_vocab,
        num_steps
    )

    tgt_array, _ = build_array(
        [tokenize(fr) for _, fr in selected_pairs],
        tgt_vocab,
        num_steps + 1,
        add_sos=True
    )

    return (
        src_array,
        tgt_array[:, :-1],
        src_valid_len,
        tgt_array[:, 1:]
    )

In [ ]:
# Demonstrate how to use this function
batch_size = 8
src, tgt, src_valid_len, label = build_arrays(
    train_pairs[0:batch_size], src_stoi, tgt_stoi, num_steps=8)
print('Mini-batch of English sentences as token IDs:')
print(src)
print('Mini-batch of French sentences as token IDs:')
print(tgt)
print('source len excluding pad:', src_valid_len.type(torch.int32))

### Exercise 1:  

Compare training time and translation quality change between batch size [5,10,32]

### 2.5 Test model final translation

In [ ]:
@torch.no_grad()
def translate(english, max_length=12):
    encoder.eval(); decoder.eval()
    source = numericalize(english, src_stoi)
    hidden = encoder(source)
    decoder_input = torch.tensor([[tgt_stoi[SOS]]], device=device)
    words = []
    for _ in range(max_length):
        logits, hidden = decoder(decoder_input, hidden)
        next_id = logits.argmax(dim=1).item()
        if next_id == tgt_stoi[EOS]:
            break
        words.append(tgt_tokens[next_id])
        decoder_input = torch.tensor([[next_id]], device=device)
    return ' '.join(words)

for english, reference in test_pairs[:8]:
    print(f'EN: {english:22s} | predicted: {translate(english):20s} | reference: {reference}')

### Exercise 2

Remove teacher forcing (feed the predicted token during training). How does convergence change?

### Exercise 3

Change RNN model to be LSTM model and see the differences. 